# 00b — `series_df` column dictionary (GSE68086_series_matrix.csv)

Companion to `00_exploration.ipynb`. **Read-only:** it does not touch `00_exploration.ipynb`,
and it never writes to `GSE68086_series_matrix.csv` / `GSE68086_TEP_data_matrix.csv`.
Run it from the `functions/` folder, like the original notebook.

**What the two datasets are.** GSE68086 is the tumour-educated platelet (TEP) RNA-seq study
(Best et al., *Cancer Cell* 2015, PMID 26525104): blood platelets from 228 patients with six
tumour types plus 55 healthy donors.

| object | file | shape | one row = | one column = |
|---|---|---|---|---|
| `data_df` | `GSE68086_TEP_data_matrix.csv` | 57 736 × 286 | one Ensembl gene (`ENSG…`, Ensembl 75 / hg19) | one sample; values are HTSeq intron-spanning read counts (raw, unnormalised) |
| `series_df` | `GSE68086_series_matrix.csv` | 285 × 46 | one sample | one GEO `!Sample_*` metadata field |

`series_df` is the *header block* of the GEO series matrix, transposed. Because GEO repeats a field
name once per value, `pandas` de-duplicates with `.1`, `.2`, … suffixes — that is where
`!Sample_characteristics_ch1.4` and friends come from. Three practical consequences, all verified below:

1. **Only 13 of the 46 columns vary across samples.** 31 are constant (study-level protocol text repeated
   once per sample, not variables) and 2 are entirely empty.
2. **The six `!Sample_characteristics_ch1*` columns are not aligned.** 245 samples use one key order,
   the 40 samples submitted later use a *rotated* order. A positional column is therefore **not** one
   variable — always parse the `"key: value"` strings by key.
3. **The join key to the counts matrix is `!Sample_source_name_ch1`, not the GSM accession.**

In [1]:
import pandas as pd
import numpy as np

SERIES = "../GSE68086_series_matrix.csv"
COUNTS = "../GSE68086_TEP_data_matrix.csv"

series_df = pd.read_csv(SERIES)          # fresh read; original notebook/objects untouched

def unquote(x):
    '''GEO values arrive triple-quoted through the CSV round-trip: "\"\"\"GSM1662534\"\"\"".'''
    return np.nan if pd.isna(x) else str(x).strip().strip('"')

series_clean = series_df.map(unquote)    # working copy, in memory only
series_df.shape

(285, 46)

## Column dictionary

`n` = number of distinct values across the 285 samples. "constant" = one value for every sample
(study-level text, safe to ignore for modelling).

| # | column | n | content |
|---|---|---|---|
| 0 | `!Sample_geo_accession` | 285 | GEO sample id. Two blocks: `GSM1662534`–`GSM1662818` (original) and `GSM1817093`–`GSM1817132` (added Jul 2015). Unique per sample, but **not** the name used in the counts matrix. |
| 1 | `!Sample_status` | 1 | Release state — `Public on Oct 30 2015`. |
| 2 | `!Sample_submission_date` | 2 | `Apr 21 2015` (245) vs `Jul 10 2015` (40). Marks the two submission waves; lines up exactly with the field-order shift in #8–13 and with the `MGH-*` sample labels. Useful as a provenance flag, *not* as a biological variable. |
| 3 | `!Sample_last_update_date` | 1 | `May 15 2019` — last GEO record edit. |
| 4 | `!Sample_type` | 1 | `SRA` — sequencing sample (as opposed to array). |
| 5 | `!Sample_channel_count` | 1 | `1`. Microarray-era field; meaningless for RNA-seq. |
| 6 | `!Sample_source_name_ch1` | 285 | Free-text lab label, e.g. `3-Breast-Her2-ampl`, `VU256-CRC`, `MGH-NSCLC-L01-TR458`, `HD-1`. **This is the column header used in `GSE68086_TEP_data_matrix.csv`, i.e. the join key to the counts.** It loosely encodes collection site (`VU`/`Vumc` = VU Amsterdam, `MGH` = Massachusetts General, `HD`/`Control` = healthy donor), an internal number and often a mutation tag — but it is unreliable as phenotype: 5 samples are labelled `Type-Unknown-*` and 4 carry a tumour tag that contradicts the `cancer type` field. Use it to align, never to derive groups. |
| 7 | `!Sample_organism_ch1` | 1 | `Homo sapiens`. |
| 8–13 | `!Sample_characteristics_ch1`, `.1` … `.5` | 41 / 5 / 247 / 12 / 5 / 11 | The six phenotype annotations, each a `"key: value"` string. **Key order differs by wave** (see below), which is why these `n` values look nonsensical. After parsing by key the six real variables are: |
| | ↳ `tissue` | 1 | `blood`. |
| | ↳ `cell type` | 2 | `Thrombocytes` / `thrombocytes` — platelets. Two spellings only because the second wave lower-cased it; normalise before grouping. |
| | ↳ `patient id` | 285 | `Breast-03`, `CRC-304`, `HC-31`, `Lung-0082`, `Panc-449`, `Chol-410`, `Liver-274`. One sample per patient — no repeated measures. The prefix is a clean redundant copy of `cancer type` (`Chol`+`Liver` → Hepatobiliary). |
| | ↳ `cancer type` | 7 | **The group / outcome variable.** Lung 60, HC 55 (healthy control), CRC 42, GBM 40, Breast 39, Pancreas 35, Hepatobiliary 14. `Lung` = NSCLC, `CRC` = colorectal, `GBM` = glioblastoma. |
| | ↳ `batch` | 6 | `Batch01`–`Batch06` (2 / 53 / 103 / 87 / 10 / 30). Library-prep + sequencing wave, and **strongly confounded with `cancer type`** (Batch05/06 are almost all Breast+Lung and contain no HC) — the reason `04_DESeq2.R` fits `~ batch + group`. |
| | ↳ `mutational subclass` | 10 | Driver-mutation annotation of the tumour: `wt` 175, `KRAS` 63, `EGFR` 13, `HER2+` 10, `EGFR, MET` 8, `Triple Negative` 6, `PIK3CA` 6, `HER2+, PIK3CA` 2, `MET` 1, `KRAS, MET` 1. Comma-separated = multiple drivers; split before use. All 55 HC are `wt`, so `wt` mixes "healthy" with "no driver found" — do not read it as a single biological state. |
| 14 | `!Sample_molecule_ch1` | 1 | `total RNA`. |
| 15 | `!Sample_extract_protocol_ch1` | 1 | Platelet isolation (EDTA tube, 120 g then 360 g spins), RNAlater, mirVana total-RNA isolation, Bioanalyzer RNA Picochip QC. |
| 16 | `!Sample_extract_protocol_ch1.1` | 1 | Library prep: 100–500 pg RNA (RIN > 7) → SMARTer Ultra Low RNA v1 amplification → Covaris shearing → TruSeq (Nano) DNA prep → 8–12 samples per lane, 100 bp single-read HiSeq 2500. The very low input explains the sparsity of the count matrix. |
| 17 | `!Sample_taxid_ch1` | 1 | `9606` (human). |
| 18 | `!Sample_description` | 3 | Empty for 278 samples; otherwise a **QC flag**: 2 samples excluded by the authors for too few mapped intron-spanning reads (`GSM1662569` / `VU258-CRC`, `GSM1662615` / `VU398-GBM`), and 5 healthy-donor samples whose two runs were merged *in silico* (`HD-3-1`, `HD-20-1`, `HD-24-2`, `HD-36`, `HD-51-1`). The two excluded samples are still present as columns in the counts matrix — filter them yourself. |
| 19 | `!Sample_data_processing` | 1 | 5′ quality trimming + adapter clipping with Trimmomatic. |
| 20 | `!Sample_data_processing.1` | 1 | STAR 2.3.0 splice-aware alignment to **hg19**, up to 10 mismatches. |
| 21 | `!Sample_data_processing.2` | 1 | Intron-spanning read selection with Picard-tools 1.115 (the study's key trick: spliced reads only). |
| 22 | `!Sample_data_processing.3` | 1 | HTSeq 0.6.1, union mode, unstranded, MAPQ ≥ 35, **Ensembl gene annotation v75** — the vintage to use when mapping `ENSG` ids to symbols. |
| 23 | `!Sample_data_processing.4` | 1 | `Genome_build: hg19`. |
| 24 | `!Sample_data_processing.5` | 1 | Supplementary-file note: all samples merged into one read-count matrix. |
| 25 | `!Sample_platform_id` | 1 | `GPL16791` = Illumina HiSeq 2500 (*Homo sapiens*). |
| 26–34 | `!Sample_contact_*` (name, email, laboratory, department, institute, address, city, zip/postal_code, country) | 1 each | Submitter block: Myron Best, Neuro-oncology Research Group, Neurosurgery, VU University Medical Center, Amsterdam. Provenance only. |
| 35 | `!Sample_data_row_count` | 1 | `0` — the series matrix carries **no** expression table; the counts live only in the supplementary file. This is why #43/#45 are empty. |
| 36 | `!Sample_instrument_model` | 1 | `Illumina HiSeq 2500`. |
| 37 | `!Sample_library_selection` | 1 | `cDNA`. |
| 38 | `!Sample_library_source` | 1 | `transcriptomic`. |
| 39 | `!Sample_library_strategy` | 1 | `RNA-Seq`. |
| 40 | `!Sample_relation` | 285 | `BioSample: …/biosample/SAMN…` — NCBI BioSample link, one per sample. |
| 41 | `!Sample_relation.1` | 285 | `SRA: …/sra?term=SRX…` — SRA experiment link; the entry point if you ever need the raw FASTQs. |
| 42 | `!Sample_supplementary_file_1` | 1 | `NONE` — no per-sample file (everything is in the one matrix). |
| 43 | `!series_matrix_table_begin` | 0 | Structural marker from the GEO text format. **All NaN** after transposing — drop it. |
| 44 | `"ID_REF"` | 285 | The expression-table header row of the original file; here just a duplicate of `!Sample_geo_accession` (#0). |
| 45 | `!series_matrix_table_end` | 0 | Structural marker. **All NaN** — drop it. |

### Bottom line
The 13 varying columns are: `!Sample_source_name_ch1` (the join key), the six
`!Sample_characteristics_ch1*` columns (which unpack into `cancer type`, `batch`, `mutational subclass`,
`patient id`, `tissue`, `cell type` — of which only the first three carry usable variation),
`!Sample_submission_date` and `!Sample_description` (provenance / QC flags), and
`!Sample_geo_accession`, `"ID_REF"` and the two `!Sample_relation` columns (external identifiers).
The other 33 columns are constant or empty.

## Verification of the claims above

In [2]:
# 1. Which columns actually vary?
prof = pd.DataFrame({
    "n_unique": series_clean.nunique(dropna=True),
    "n_missing": series_clean.isna().sum(),
    "example": series_clean.apply(lambda c: str(c.dropna().iloc[0])[:70] if c.notna().any() else ""),
})
print("varying columns :", (prof.n_unique > 1).sum())
print("constant columns:", (prof.n_unique == 1).sum())
print("all-NaN columns :", (prof.n_unique == 0).sum())
prof[prof.n_unique > 1]

varying columns : 13
constant columns: 31
all-NaN columns : 2


,n_unique,n_missing,example
!Sample_geo_accession,285,0,GSM1662534
!Sample_submission_date,2,0,Apr 21 2015
!Sample_source_name_ch1,285,0,3-Breast-Her2-ampl
!Sample_characteristics_ch1,41,0,tissue: blood
!Sample_characteristics_ch1.1,5,0,cell type: Thrombocytes
!Sample_characteristics_ch1.2,247,0,patient id: Breast-03
!Sample_characteristics_ch1.3,12,0,cancer type: Breast
!Sample_characteristics_ch1.4,5,0,batch: Batch03
!Sample_characteristics_ch1.5,11,0,mutational subclass: HER2+
!Sample_description,3,0,


In [3]:
# 2. The characteristics columns hold different KEYS in different rows.
ch_cols = [c for c in series_clean.columns if c.startswith("!Sample_characteristics_ch1")]

key_of = lambda s: s.split(":")[0].strip() if isinstance(s, str) and ":" in s else "<no key>"
key_pattern = series_clean[ch_cols].apply(lambda row: tuple(key_of(v) for v in row), axis=1)

print(key_pattern.value_counts().to_string(), "\n")
# ...and the 40 rotated rows are exactly the second submission wave:
print(pd.crosstab(series_clean["!Sample_submission_date"],
                  key_pattern.map(lambda p: "rotated" if p[0] != "tissue" else "original")))

(tissue, cell type, patient id, cancer type, batch, mutational subclass)    245
(patient id, cancer type, batch, mutational subclass, tissue, cell type)     40 

col_0                    original  rotated
!Sample_submission_date                   
Apr 21 2015                   245        0
Jul 10 2015                     0       40


In [4]:
# 3. Parse by key -> tidy sample sheet: the same columns as 01_sample sheet creation.R, plus gsm.
#    (Mirrors what GEOquery's pData() hands the R scripts, which is why
#     01_sample sheet creation.R is unaffected by the key-order shift.)
def parse_characteristics(row):
    out = {}
    for v in row:
        if isinstance(v, str) and ":" in v:
            k, _, val = v.partition(":")
            out[k.strip()] = val.strip()
    return pd.Series(out)

chars = series_clean[ch_cols].apply(parse_characteristics, axis=1).rename(
    columns={"cancer type": "group", "patient id": "patient", "mutational subclass": "subclass"}
)

series_tidy = pd.concat(
    [
        series_clean["!Sample_source_name_ch1"].rename("sample"),   # join key to the counts
        series_clean["!Sample_geo_accession"].rename("gsm"),
        chars[["batch", "group", "subclass", "patient"]],
    ],
    axis=1,
)

# Deliberately left out: 'tissue' (always blood) and 'cell type' (always thrombocytes) are constant,
# and 'submission_date' / the !Sample_description note stay in series_clean if ever needed.
print(list(series_tidy.columns))
series_tidy.head()

['sample', 'gsm', 'batch', 'group', 'subclass', 'patient']


,sample,gsm,batch,group,subclass,patient
0,3-Breast-Her2-ampl,GSM1662534,Batch03,Breast,HER2+,Breast-03
1,8-Breast-WT,GSM1662535,Batch03,Breast,wt,Breast-08
2,10-Breast-Her2-ampl,GSM1662536,Batch03,Breast,HER2+,Breast-10
3,Breast-100,GSM1662537,Batch04,Breast,Triple Negative,Breast-100
4,15-Breast-Her2-ampl,GSM1662538,Batch03,Breast,HER2+,Breast-15


In [12]:
# 4. The variables, and the batch/group confounding that 04_DESeq2.R models with ~ batch + group
for col in ["group", "batch", "subclass"]:
    print(f"--- {col} ---")
    print(series_tidy[col].value_counts().to_string(), "\n")

print("patients:", series_tidy.patient.nunique(), "unique for", len(series_tidy), "samples",
      "-> one sample per patient\n")
crosstab = pd.crosstab(series_tidy.group, series_tidy.batch, margins=True)
crosstab

--- group ---
group
Lung             60
HC               55
CRC              42
GBM              40
Breast           39
Pancreas         35
Hepatobiliary    14 

--- batch ---
batch
Batch03    103
Batch04     87
Batch02     53
Batch06     30
Batch05     10
Batch01      2 

--- subclass ---
subclass
wt                 175
KRAS                63
EGFR                13
HER2+               10
EGFR, MET            8
Triple Negative      6
PIK3CA               6
HER2+, PIK3CA        2
MET                  1
KRAS, MET            1 

patients: 285 unique for 285 samples -> one sample per patient



batch,Batch01,Batch02,Batch03,Batch04,Batch05,Batch06,All
group,,,,,,,
Breast,0,0,15,9,6,9,39
CRC,0,4,21,13,1,3,42
GBM,2,12,15,11,0,0,40
HC,0,16,15,24,0,0,55
Hepatobiliary,0,1,7,5,0,1,14
Lung,0,19,7,14,3,17,60
Pancreas,0,1,23,11,0,0,35
All,2,53,103,87,10,30,285


In [25]:
crosstab_batchslice = crosstab.iloc[:,1:4]
crosstab_batchslice["All"] = crosstab_batchslice.sum(axis=1)
crosstab_batchslice

batch,Batch02,Batch03,Batch04,All
group,,,,
Breast,0,15,9,24
CRC,4,21,13,38
GBM,12,15,11,38
HC,16,15,24,55
Hepatobiliary,1,7,5,13
Lung,19,7,14,40
Pancreas,1,23,11,35
All,53,103,87,243


In [26]:
# 5. The join key is sample, NOT the GSM accession.
counts_cols = pd.read_csv(COUNTS, nrows=0).columns[1:]      # header only; the file is 35 MB

print("counts matrix samples :", len(counts_cols))
print("matched by sample:", series_tidy["sample"].isin(counts_cols).sum())
print("matched by GSM        :", series_tidy["gsm"].isin(counts_cols).sum())
print("set-equal on source_name:", set(counts_cols) == set(series_tidy["sample"]))

counts matrix samples : 285
matched by sample: 285
matched by GSM        : 0
set-equal on source_name: True


In [27]:
# 6. QC flags hiding in !Sample_description -- and the fact that they are NOT pre-removed.
#    The note is not part of series_tidy, so pull it from series_clean (same row order).
note = series_clean["!Sample_description"].fillna("").rename("note")
flags = series_tidy.join(note)
flags = flags[flags.note != ""]

print(flags.note.str.slice(0, 60).value_counts().to_string(), "\n")
excluded = flags[flags.note.str.contains("did not yield sufficient")]
print("authors excluded these:", excluded["sample"].tolist())
print("still columns in the counts matrix:", excluded["sample"].isin(counts_cols).all())
flags[["sample", "gsm", "group", "batch", "note"]]

note
Please note that the mapped and counted intron-spanning read    5
Please note this sample did not yield sufficient number of t    2 

authors excluded these: ['VU258-CRC', 'VU398-GBM']
still columns in the counts matrix: True


,sample,gsm,group,batch,note
35,VU258-CRC,GSM1662569,CRC,Batch02,Please note this sample did not yield sufficie...
81,VU398-GBM,GSM1662615,GBM,Batch02,Please note this sample did not yield sufficie...
114,HD-3-1,GSM1662648,HC,Batch03,Please note that the mapped and counted intron...
126,HD-20-1,GSM1662660,HC,Batch03,Please note that the mapped and counted intron...
146,HD-24-2,GSM1662680,HC,Batch04,Please note that the mapped and counted intron...
156,HD-36,GSM1662690,HC,Batch04,Please note that the mapped and counted intron...
164,HD-51-1,GSM1662698,HC,Batch04,Please note that the mapped and counted intron...


In [ ]:
# 6b. Sequencing depth -- DERIVED FROM THE COUNTS MATRIX, not from series_df.
#     series_df carries no depth field at all (!Sample_data_row_count is literally "0"),
#     so this is the one sample-level number the metadata cannot give you.
#     Reads the full 35 MB matrix: the slow cell of this notebook.
counts = pd.read_csv(COUNTS, index_col=0)

count          285
mean     2,091,098
std      1,017,867
min        218,517
25%      1,299,744
50%      2,015,926
75%      2,802,022
max      7,273,295

max/min depth ratio : 33.3 x
all-zero genes      : 29195 of 57736
median share of genes detected per sample: 0.15


,lib_size,genes_detected,depth_pctile,group,batch
sample,,,,,
VU258-CRC,218517,2918,0.4,CRC,Batch02
MGH-BrCa-H-59,383323,6613,0.7,Breast,Batch05
VU398-GBM,396494,4643,1.1,GBM,Batch02
HD-27-2,422514,8629,1.4,HC,Batch04
HD-30-2,433762,9271,1.8,HC,Batch04
HD-25-2,434795,9701,2.1,HC,Batch04
HD-48-1,448816,10503,2.5,HC,Batch04
HD-24-2,450316,8279,2.8,HC,Batch04


In [34]:
qc = pd.DataFrame({
    "lib_size": counts.sum(axis=0),                   # total intron-spanning reads per sample
    "genes_detected": (counts > 0).sum(axis=0),       # non-zero genes, out of 57,736
}).rename_axis("sample")

print(qc.lib_size.describe().apply("{:,.0f}".format).to_string())
print("\nmax/min depth ratio :", round(qc.lib_size.max() / qc.lib_size.min(), 1), "x")
print("all-zero genes      :", int((counts.sum(axis=1) == 0).sum()), "of", counts.shape[0])
print("median share of genes detected per sample:",
      round(float((qc.genes_detected / counts.shape[0]).median()), 3))

# Kept separate from series_tidy on purpose: series_tidy is metadata straight out of the
# series matrix, qc is computed from the counts. Merge with series_tidy.join(qc, on="sample").
qc["depth_pctile"] = qc.lib_size.rank(pct=True).mul(100).round(1)
qc_series = qc.join(series_tidy.set_index("sample")[["group", "batch"]])
qc_series.nsmallest(8, "lib_size")

count          285
mean     2,091,098
std      1,017,867
min        218,517
25%      1,299,744
50%      2,015,926
75%      2,802,022
max      7,273,295

max/min depth ratio : 33.3 x
all-zero genes      : 29195 of 57736
median share of genes detected per sample: 0.15


,lib_size,genes_detected,depth_pctile,group,batch
sample,,,,,
VU258-CRC,218517,2918,0.4,CRC,Batch02
MGH-BrCa-H-59,383323,6613,0.7,Breast,Batch05
VU398-GBM,396494,4643,1.1,GBM,Batch02
HD-27-2,422514,8629,1.4,HC,Batch04
HD-30-2,433762,9271,1.8,HC,Batch04
HD-25-2,434795,9701,2.1,HC,Batch04
HD-48-1,448816,10503,2.5,HC,Batch04
HD-24-2,450316,8279,2.8,HC,Batch04


In [38]:
qc_series

,lib_size,genes_detected,depth_pctile,group,batch
sample,,,,,
3-Breast-Her2-ampl,3897719,10314,96.1,Breast,Batch03
8-Breast-WT,3483473,8931,92.3,Breast,Batch03
10-Breast-Her2-ampl,3317403,9006,89.1,Breast,Batch03
Breast-100,1027235,7214,16.1,Breast,Batch04
15-Breast-Her2-ampl,2718077,8749,71.6,Breast,Batch03
...,...,...,...,...,...
MGH-NSCLC-L40-TR520,1197141,4522,22.1,Lung,Batch06
MGH-NSCLC-L51-TR521,2159531,7740,54.0,Lung,Batch06
MGH-NSCLC-L58-TR525,2888997,7737,78.9,Lung,Batch06


In [9]:
# 7. Why source_name must not be used as a phenotype: its embedded tumour tag disagrees
#    with the annotated cancer type for 4 samples, and 5 samples have no tag at all.
tag = {"Breast": "Breast", "BrCa": "Breast", "CRC": "CRC", "GBM": "GBM",
       "Pancr": "Pancreas", "Panc": "Pancreas", "Liver": "Hepatobiliary", "Chol": "Hepatobiliary",
       "NSCLC": "Lung", "Lung": "Lung", "HD": "HC", "Control": "HC"}

def tag_of(name):
    for k, v in tag.items():
        if k.lower() in name.lower():
            return v
    return None

chk = series_tidy.assign(label_says=series_tidy["sample"].map(tag_of))
print("no recognisable tumour tag:", chk.label_says.isna().sum(), "samples ->",
      chk.loc[chk.label_says.isna(), "sample"].tolist())
chk.loc[chk.label_says.notna() & (chk.label_says != chk.group),
        ["sample", "patient", "group", "label_says"]]

no recognisable tumour tag: 7 samples -> ['Type-Unknown-6', 'Type-Unknown-1', 'Type-Unknown-5', 'VU383Platelet-hiseq', 'VU394Platelet-hiseq', 'Type-Unknown-3', 'Type-Unknown-4']


,sample,patient,group,label_says
167,VU274-CRC,Liver-274,Hepatobiliary,CRC
194,VU271-CRC,Lung-271,Lung,CRC
210,VU260-CRC,Panc-260,Pancreas,CRC
260,MGH-CRC-BRAF4-TR547,Chol-BRAF4,Hepatobiliary,CRC


## Gotchas to carry into the next steps

- **Never index the characteristics positionally.** `series_df["!Sample_characteristics_ch1.4"]` is
  `batch` for 245 samples and `tissue: blood` for the other 40 — the `groupby` in
  `00_exploration.ipynb` cell 10 is grouping a mix of the two. Parse by key (cell 3 above).
  The R side is safe: `GEOquery::pData()` already resolves fields by name, so
  `raw$"batch:ch1"` in `01_sample sheet creation.R` is correct.
- **Align counts on `!Sample_source_name_ch1`.** `01_sample sheet creation.R` does exactly this
  (`stopifnot(setequal(rownames(ss), colnames(counts)))`); keep that check.
- **`batch` and `group` are confounded** and Batch05/Batch06 hold no healthy controls, so any
  HC-vs-cancer contrast inside those batches is unidentifiable. `~ batch + group` handles the rest.
- **Drop the 2 author-excluded samples** (`VU258-CRC`, `VU398-GBM`) — they are still in the matrix.
- **`wt` in `subclass` is not one state**: it covers healthy donors and tumours with no detected driver.
- **`submission_date` is a real technical covariate** (second wave = MGH samples, rotated metadata,
  lower-cased cell type), but it is nested in `batch`, so do not add both to a model.
- **`series_tidy` holds only `sample`, `gsm`, `batch`, `group`, `subclass`, `patient`** — the columns
  of `01_sample sheet creation.R` plus the GSM id. `tissue` and `cell type` are constant and were
  dropped; `submission_date` and the `!Sample_description` note live on in `series_clean`.
- **Depth is not metadata.** `lib_size` / `genes_detected` exist only in the counts matrix
  (`!Sample_data_row_count` is `0`). Depth spans 33x across samples, and the two author-excluded
  samples sit at the very bottom of it — which is what makes that QC flag worth acting on.
- Counts are **raw intron-spanning reads** on hg19 / Ensembl 75 — feed them to DESeq2/edgeR as-is,
  never pre-normalised.